# Analyse métier - customers_data

Notebook d'exploration : lecture CSV, enrichissement via taux de change, synthèse et graphique Altair.

In [8]:
import altair as alt
import polars as pl
from main import build_enriched_report, load_exchange_rates, load_orders


In [9]:
rates_df, fallback_rate = load_exchange_rates("input/daily.csv")
orders_df = load_orders("input/restaurant_orders_anonymous.csv")
report_df = build_enriched_report(orders_df, rates_df, fallback_rate or 1.0)
report_df.head()

Order ID,Customer Name,Food Item,Category,Quantity,Price,Payment Method,Order Time,order_date_fr,price_eur,daily_revenue_eur
i64,str,str,str,i64,f64,str,str,str,f64,f64
2268,"""Mary Vega DDS""","""Pasta""","""Main""",5,16.52,"""Cash""","""2025-02-02 14:28:41""","""02/02/2025""",14.23,14.23
3082,"""Brandon Myers""","""Brownie""","""Dessert""",4,17.27,"""Debit Card""","""2025-06-08 10:57:47""","""08/06/2025""",14.88,14.88
3160,"""Margaret Wells""","""Pasta""","""Main""",1,3.37,"""Credit Card""","""2025-03-04 07:41:41""","""04/03/2025""",3.2,3.2
1272,"""Michael Matthews""","""Pasta""","""Main""",5,2.2,"""Online Payment""","""2025-05-15 12:43:45""","""15/05/2025""",1.97,1.97


In [10]:
report_df.schema

Schema([('Order ID', Int64),
        ('Customer Name', String),
        ('Food Item', String),
        ('Category', String),
        ('Quantity', Int64),
        ('Price', Float64),
        ('Payment Method', String),
        ('Order Time', String),
        ('order_date_fr', String),
        ('price_eur', Float64),
        ('daily_revenue_eur', Float64)])

In [11]:
category_summary = (
    report_df.group_by("Category")
    .agg([
        pl.len().alias("nb_commandes"),
        pl.col("price_eur").sum().round(2).alias("revenue_eur"),
    ])
    .sort("revenue_eur", descending=True)
)
category_summary

Category,nb_commandes,revenue_eur
str,u64,f64
"""Main""",3,19.4
"""Dessert""",1,14.88


In [19]:
alt.Chart(category_summary).mark_bar().encode(
    x=alt.X("Category:N", title="Catégorie"),
    y=alt.Y("revenue_eur:Q", title="Revenu EUR"),
    tooltip=["Category", "nb_commandes", "revenue_eur"],
).properties(title="Revenu par catégorie", width=500)

alt.Chart(...)